# **ClinTrialPredict Estimation: Planned Site Count Benchmark Layer (S1B/S2)**

This notebook is the central reproducible analytical notebook for the `planned_sites` benchmark foundation. It explains and audits the deterministic site-count benchmark foundation created across S1B and S2.

### **Key S1B/S2 objectives:**
1. Keep S2 backend/artifact-only: do not activate `planned_sites` in Simulation Mode.
2. Use `number_of_facilities` only as a registry-derived aggregate facility-count proxy, not as true planned sites, true actual activated sites, or true estimated sites.
3. Build historical site-count proxy benchmarks from completed trials with positive `number_of_facilities`.
4. Use a deterministic four-level fallback hierarchy and percentile bands, not a site-count ML model.
5. Produce and inspect the compact runtime artifact that can later be used by the app image.
6. Demonstrate runtime lookup and metadata construction without changing XGBoost, SHAP, audit parity, UI, API contracts, prediction payloads, model artifacts, taxonomy artifacts, or deployment configuration.
7. Explore patients-per-site coherence as a future secondary signal for keeping `planned_sites` realistic relative to `planned_enrollment`, without activating that signal in S2.

The benchmark is a reference signal for a later Simulation Mode / narrative / Coherence layer. It does not modify the Completion Score.

## **Reproducibility Workflow**

To reproduce the artifact, the source-of-truth command is:

```bash
python scripts/build_site_benchmarks.py
```

That rebuilds:

```text
frontend/data/site_benchmarks_v1.csv
frontend/data/site_benchmarks_v1_report.json
```

Then this notebook can be run to inspect and validate the regenerated outputs.

Current intended workflow:

1. Confirm S1B source contract and caveat:

   ```text
   number_of_facilities = registry-derived aggregate facility-count proxy
   not true planned sites
   not true actual activated sites
   not true estimated sites
   ```

2. Rebuild artifact:

   ```bash
   python scripts/build_site_benchmarks.py
   ```

3. Validate artifact and runtime utility:

   ```bash
   python scripts/check_site_benchmarks.py
   ```

4. Open/run notebook:

   ```text
   notebooks/estimation_sites.ipynb
   ```

5. Do not activate UI until a separate S3 implementation is explicitly authorized:

   ```text
   planned_sites remains inactive
   ACTIVE_OPERATIONAL_ASSUMPTION_KEYS remains unchanged
   frontend/views/edit_trial.py remains unchanged in S2
   ```

The notebook also reproduces the calculations in memory in sections such as:

```text
SITES_BENCHMARK_COHORTS
SITES_BENCHMARK_PERCENTILES
SITES_RUNTIME_LOOKUP_DEMO
SITES_ENROLLMENT_ALIGNMENT
SITES_VALIDATION
```


#### <REF:ENV_CONFIG>
> #### **1. Development Environment Configuration**
>
> Configure notebook behavior, reproducibility defaults, warning filters, display settings, and lightweight helper imports.


In [12]:
# <REF:ENV_CONFIG_CODE>
import json
import math
import random
import tempfile
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)
random.seed(42)
np.random.seed(42)
# <REF:/ENV_CONFIG_CODE>


#### <REF:PATH_RESOLUTION>
> #### **2. Project Path Resolution and Artifact Locations**
>
> Resolve the repository root dynamically, add it to `PYTHONPATH`, and define source, artifact, report, builder, checker, and runtime paths.


In [13]:
# <REF:PATH_RESOLUTION_CODE>
import sys

current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "src").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not resolve project root containing src/")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_CLINPRED_PATH = project_root / "data" / "data_clinpred.csv"
CALCULATED_VALUES_PATH = project_root / "data" / "calculated_values.txt"
ARTIFACT_PATH = project_root / "frontend" / "data" / "site_benchmarks_v1.csv"
REPORT_PATH = project_root / "frontend" / "data" / "site_benchmarks_v1_report.json"
S1B_AUDIT_PATH = project_root / "notebooks" / "outputs" / "site_count_s1b_audit.json"
BUILDER_PATH = project_root / "scripts" / "build_site_benchmarks.py"
CHECKER_PATH = project_root / "scripts" / "check_site_benchmarks.py"
RUNTIME_UTILITY_PATH = project_root / "src" / "site_benchmarks.py"

print("Project root:", project_root)
print("Source data:", DATA_CLINPRED_PATH.relative_to(project_root))
print("Compact artifact:", ARTIFACT_PATH.relative_to(project_root))
print("Report JSON:", REPORT_PATH.relative_to(project_root))
# <REF:/PATH_RESOLUTION_CODE>


Project root: /home/delaunan/code/delaunan/clintrialpredict
Source data: data/data_clinpred.csv
Compact artifact: frontend/data/site_benchmarks_v1.csv
Report JSON: frontend/data/site_benchmarks_v1_report.json


#### <REF:DATA_LOAD>
> #### **3. Source Data Load**
>
> Load `data/data_clinpred.csv`, confirm fields needed by the S2 builder exist, and inspect `data/calculated_values.txt` when present.


In [14]:
# <REF:DATA_LOAD_CODE>
if not DATA_CLINPRED_PATH.exists():
    raise FileNotFoundError(f"Missing source data: {DATA_CLINPRED_PATH}")

df_full = pd.read_csv(DATA_CLINPRED_PATH, low_memory=False)
print(f"Loaded data_clinpred rows: {len(df_full):,}")
print(f"Loaded data_clinpred columns: {len(df_full.columns):,}")

required_site_columns = [
    "nct_id",
    "number_of_facilities",
    "overall_status",
    "phase_ml",
    "phase",
    "therapeutic_area",
    "therapeutic_area_ml",
    "gbd_cause_id_3_ml",
    "gbd_indication_name_3",
    "is_rare_disease_ml",
    "is_rare_disease",
    "enrollment",
    "enrollment_type",
]
missing_site_columns = [column for column in required_site_columns if column not in df_full.columns]
print("Missing required/inspection columns:", missing_site_columns)
assert not [column for column in ["nct_id", "number_of_facilities", "overall_status", "phase", "therapeutic_area", "gbd_cause_id_3_ml", "is_rare_disease_ml"] if column not in df_full.columns]

if CALCULATED_VALUES_PATH.exists():
    calculated_header = pd.read_csv(CALCULATED_VALUES_PATH, sep="|", nrows=0).columns.tolist()
    print("calculated_values.txt header:")
    print(calculated_header)
else:
    print("calculated_values.txt not found in this checkout")
# <REF:/DATA_LOAD_CODE>


Loaded data_clinpred rows: 34,066
Loaded data_clinpred columns: 157
Missing required/inspection columns: []
calculated_values.txt header:
['id', 'nct_id', 'number_of_facilities', 'number_of_nsae_subjects', 'number_of_sae_subjects', 'registered_in_calendar_year', 'nlm_download_date', 'actual_duration', 'were_results_reported', 'months_to_report_results', 'has_us_facility', 'has_single_facility', 'minimum_age_num', 'maximum_age_num', 'minimum_age_unit', 'maximum_age_unit', 'number_of_primary_outcomes_to_measure', 'number_of_secondary_outcomes_to_measure', 'number_of_other_outcomes_to_measure']


#### <REF:SITES_SOURCE_CONTRACT>
> #### **4. Site-Count Source Contract**
>
> Confirm the S1B source contract: `number_of_facilities` is available and numeric, comes from `calculated_values.txt`, and has no planned/actual/estimated qualifier. The benchmark must describe it as a registry-derived aggregate facility-count proxy.


In [15]:
# <REF:SITES_SOURCE_CONTRACT_CODE>
df = df_full.copy()
df["number_of_facilities"] = pd.to_numeric(df["number_of_facilities"], errors="coerce")
df["enrollment"] = pd.to_numeric(df.get("enrollment"), errors="coerce")
df["gbd_cause_id_3_ml"] = pd.to_numeric(df["gbd_cause_id_3_ml"], errors="coerce").fillna(0).astype(int)
df["is_rare_disease_ml"] = pd.to_numeric(df["is_rare_disease_ml"], errors="coerce").fillna(0).astype(int)

for col in ["overall_status", "phase", "therapeutic_area"]:
    df[col] = df[col].fillna("UNKNOWN").astype(str).str.strip().str.upper().replace({"": "UNKNOWN"})

site_qualifier_like = [
    col for col in df.columns
    if any(token in col.lower() for token in ["site", "facilit"])
]

local_facility_like_files = sorted(
    str(path.relative_to(project_root))
    for path in (project_root / "data").glob("**/*")
    if path.is_file() and any(token in path.name.lower() for token in ["facility", "facilities", "site"])
)

source_contract = {
    "number_of_facilities_exists": "number_of_facilities" in df.columns,
    "number_of_facilities_present": int(df["number_of_facilities"].notna().sum()),
    "number_of_facilities_missing": int(df["number_of_facilities"].isna().sum()),
    "source_origin": "data/calculated_values.txt merged by src/prep/data_loader_clinpred.py::_engineer_facilities(...) with ['nct_id', 'number_of_facilities']",
    "interpretation": "registry-derived aggregate facility-count proxy; not true planned/actual/estimated site count",
    "site_or_facility_like_columns": site_qualifier_like,
    "local_facility_or_site_like_files": local_facility_like_files,
}
print(json.dumps(source_contract, indent=2))
# <REF:/SITES_SOURCE_CONTRACT_CODE>


{
  "number_of_facilities_exists": true,
  "number_of_facilities_present": 34066,
  "number_of_facilities_missing": 0,
  "source_origin": "data/calculated_values.txt merged by src/prep/data_loader_clinpred.py::_engineer_facilities(...) with ['nct_id', 'number_of_facilities']",
  "interpretation": "registry-derived aggregate facility-count proxy; not true planned/actual/estimated site count",
  "site_or_facility_like_columns": [
    "number_of_facilities"
  ],
  "local_facility_or_site_like_files": []
}


#### <REF:SITES_QUALITY_AUDIT>
> #### **5. Site-Count Field Audit**
>
> Recalculate quality statistics and grouped summaries for `number_of_facilities`. The field has a long upper tail, so percentile benchmarks are preferred over mean-based benchmarks.


In [16]:
# <REF:SITES_QUALITY_AUDIT_CODE>
def numeric_summary(series: pd.Series) -> dict:
    clean = pd.to_numeric(series, errors="coerce")
    return {
        "total_rows": int(len(clean)),
        "present": int(clean.notna().sum()),
        "missing": int(clean.isna().sum()),
        "zero": int(clean.eq(0).sum()),
        "negative": int(clean.lt(0).sum()),
        "positive": int(clean.gt(0).sum()),
        "min": float(clean.min()),
        "p25": float(clean.quantile(0.25)),
        "median": float(clean.quantile(0.50)),
        "p75": float(clean.quantile(0.75)),
        "p90": float(clean.quantile(0.90)),
        "p95": float(clean.quantile(0.95)),
        "p99": float(clean.quantile(0.99)),
        "max": float(clean.max()),
    }

def grouped_facility_summary(frame: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for key, group in frame.groupby(group_col, dropna=False):
        vals = group["number_of_facilities"]
        rows.append({
            "group": key,
            "rows": int(len(group)),
            "present": int(vals.notna().sum()),
            "zero": int(vals.eq(0).sum()),
            "positive": int(vals.gt(0).sum()),
            "median": float(vals.median()),
            "p75": float(vals.quantile(0.75)),
            "p90": float(vals.quantile(0.90)),
            "max": float(vals.max()),
        })
    return pd.DataFrame(rows).sort_values("rows", ascending=False).reset_index(drop=True)

site_quality = numeric_summary(df["number_of_facilities"])
print(json.dumps(site_quality, indent=2))

print("By status:")
display(grouped_facility_summary(df, "overall_status"))
print("By phase:")
display(grouped_facility_summary(df, "phase"))
print("By therapeutic area, top 20 by rows:")
display(grouped_facility_summary(df, "therapeutic_area").head(20))
# <REF:/SITES_QUALITY_AUDIT_CODE>


{
  "total_rows": 34066,
  "present": 34066,
  "missing": 0,
  "zero": 2242,
  "negative": 0,
  "positive": 31824,
  "min": 0.0,
  "p25": 1.0,
  "median": 12.0,
  "p75": 40.0,
  "p90": 93.0,
  "p95": 149.0,
  "p99": 302.34999999999854,
  "max": 1745.0
}
By status:


,group,rows,present,zero,positive,median,p75,p90,max
0,COMPLETED,20719,20719,839,19880,13.0,40.0,90.0,1611.0
1,TERMINATED,4195,4195,116,4079,14.0,41.0,92.0,1745.0
2,RECRUITING,4134,4134,0,4134,10.0,39.0,104.0,1264.0
3,ACTIVE_NOT_RECRUITING,2487,2487,2,2485,28.0,79.0,175.0,1065.0
4,WITHDRAWN,1245,1245,685,560,0.0,1.0,9.6,311.0
5,NOT_YET_RECRUITING,1139,1139,600,539,0.0,1.0,5.2,226.0
6,ENROLLING_BY_INVITATION,147,147,0,147,10.0,30.5,92.8,371.0


By phase:


,group,rows,present,zero,positive,median,p75,p90,max
0,PHASE2,14387,14387,881,13506,9.0,28.00,56.0,456.0
1,PHASE3,13884,13884,1046,12838,25.0,74.00,157.0,1745.0
2,PHASE1/PHASE2,4739,4739,240,4499,5.0,14.00,28.0,166.0
3,PHASE2/PHASE3,1056,1056,75,981,13.0,43.25,91.0,577.0


By therapeutic area, top 20 by rows:


,group,rows,present,zero,positive,median,p75,p90,max
0,ONCOLOGY,8396,8396,505,7891,13.0,42.00,117.0,794.0
1,INFECTIONS,3158,3158,231,2927,6.0,24.00,58.0,464.0
2,NEUROLOGY,2599,2599,155,2444,16.0,48.00,94.0,495.0
3,DERMATOLOGY,2414,2414,139,2275,10.0,34.00,72.0,547.0
4,GASTROINTESTINAL,2290,2290,221,2069,15.0,50.00,115.0,766.0
5,METABOLIC,2110,2110,149,1961,15.0,50.75,108.1,888.0
6,RESPIRATORY,2001,2001,149,1852,12.0,53.00,123.0,1611.0
7,MUSCULOSKELETAL,1972,1972,129,1843,13.0,42.25,93.0,507.0
8,CARDIOVASCULAR,1670,1670,116,1554,13.0,50.75,138.1,1745.0
9,OPHTHALMOLOGY,1574,1574,121,1453,5.0,23.00,57.0,320.0


#### <REF:SITES_FLAGS>
> #### **6. Source and Readiness Flags**
>
> Reproduce the S2 source/readiness flags using the same implementation functions as the production builder.


In [17]:
# <REF:SITES_FLAGS_CODE>
from scripts.build_site_benchmarks import add_source_flags, load_source

benchmark_source = add_source_flags(load_source(DATA_CLINPRED_PATH))
flag_columns = [
    "is_completed_positive_site_count_target",
    "is_current_registry_facility_count_proxy",
]

print("Benchmark source rows:", f"{len(benchmark_source):,}")
print("Flag counts:")
display(benchmark_source[flag_columns].sum().rename("rows").reset_index().rename(columns={"index": "flag"}))

print("Status and positivity summary:")
display(
    benchmark_source.groupby("overall_status")
    .agg(
        rows=("nct_id", "size"),
        positive_facility_proxy=("number_of_facilities", lambda x: int(pd.to_numeric(x, errors="coerce").gt(0).sum())),
        median_facility_proxy=("number_of_facilities", "median"),
        p90_facility_proxy=("number_of_facilities", lambda x: float(pd.to_numeric(x, errors="coerce").quantile(0.90))),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)
# <REF:/SITES_FLAGS_CODE>


Benchmark source rows: 34,066
Flag counts:


,flag,rows
0,is_completed_positive_site_count_target,19880
1,is_current_registry_facility_count_proxy,7305


Status and positivity summary:


,overall_status,rows,positive_facility_proxy,median_facility_proxy,p90_facility_proxy
1,COMPLETED,20719,19880,13.0,90.0
5,TERMINATED,4195,4079,14.0,92.0
4,RECRUITING,4134,4134,10.0,104.0
0,ACTIVE_NOT_RECRUITING,2487,2485,28.0,175.0
6,WITHDRAWN,1245,560,0.0,9.6
3,NOT_YET_RECRUITING,1139,539,0.0,5.2
2,ENROLLING_BY_INVITATION,147,147,10.0,92.8


#### <REF:SITES_BENCHMARK_TARGET>
> #### **7. Benchmark Target**
>
> S2 uses completed trials with `number_of_facilities > 0` as the benchmark target. This target represents completed registry facility-count proxy values, not true actual activated site counts.


In [18]:
# <REF:SITES_BENCHMARK_TARGET_CODE>
completed_positive = benchmark_source.loc[benchmark_source["is_completed_positive_site_count_target"]].copy()
ongoing_proxy = benchmark_source.loc[benchmark_source["is_current_registry_facility_count_proxy"]].copy()

benchmark_target_summary = {
    "completed_positive_site_count_targets": int(len(completed_positive)),
    "ongoing_current_registry_facility_count_proxy_rows": int(len(ongoing_proxy)),
    "target_definition": "overall_status == COMPLETED and number_of_facilities > 0",
    "source_caveat": "completed registry facility-count proxy; not true actual activated sites",
}
print(json.dumps(benchmark_target_summary, indent=2))

print("Top 20 largest facility-count proxy values:")
display(
    benchmark_source.sort_values("number_of_facilities", ascending=False)
    [["nct_id", "number_of_facilities", "phase", "therapeutic_area", "overall_status"]]
    .head(20)
    .reset_index(drop=True)
)
# <REF:/SITES_BENCHMARK_TARGET_CODE>


{
  "completed_positive_site_count_targets": 19880,
  "ongoing_current_registry_facility_count_proxy_rows": 7305,
  "target_definition": "overall_status == COMPLETED and number_of_facilities > 0",
  "source_caveat": "completed registry facility-count proxy; not true actual activated sites"
}
Top 20 largest facility-count proxy values:


,nct_id,number_of_facilities,phase,therapeutic_area,overall_status
0,NCT01975376,1745,PHASE3,CARDIOVASCULAR,TERMINATED
1,NCT01313676,1611,PHASE3,RESPIRATORY,COMPLETED
2,NCT01975389,1595,PHASE3,CARDIOVASCULAR,TERMINATED
3,NCT01663402,1388,PHASE3,CARDIOVASCULAR,COMPLETED
4,NCT02993406,1319,PHASE3,CARDIOVASCULAR,COMPLETED
5,NCT01764633,1284,PHASE3,CARDIOVASCULAR,COMPLETED
6,NCT07000357,1264,PHASE3,CARDIOVASCULAR,RECRUITING
7,NCT01991795,1237,PHASE3,CARDIOVASCULAR,COMPLETED
8,NCT01126437,1191,PHASE3,RESPIRATORY,COMPLETED
9,NCT07064473,1150,PHASE3,CARDIOVASCULAR,RECRUITING


#### <REF:SITES_BENCHMARK_COHORTS>
> #### **8. Benchmark Cohorts and Fallback Hierarchy**
>
> Build in-memory benchmark cohorts with the approved S2 hierarchy and `min_n = 50` confidence threshold. This mirrors the production builder before inspecting the compact artifact.


In [19]:
# <REF:SITES_BENCHMARK_COHORTS_CODE>
from scripts.build_site_benchmarks import build_benchmarks

MIN_N = 50
in_memory_artifact = build_benchmarks(
    benchmark_source,
    min_n=MIN_N,
    source_data_version="notebook_reproduction",
    created_at="notebook_reproduction",
)

hierarchy_summary = (
    in_memory_artifact.groupby("benchmark_level_used")
    .agg(
        rows=("benchmark_key", "size"),
        confident_rows=("low_confidence_flag", lambda x: int((~x).sum())),
        low_confidence_rows=("low_confidence_flag", lambda x: int(x.sum())),
        duplicate_keys=("benchmark_key", lambda x: int(x.duplicated().sum())),
        min_p50=("benchmark_p50", "min"),
        max_p50=("benchmark_p50", "max"),
    )
    .reset_index()
)
display(hierarchy_summary)

print("Total in-memory artifact rows:", len(in_memory_artifact))
print("Duplicate benchmark keys:", int(in_memory_artifact["benchmark_key"].duplicated().sum()))
# <REF:/SITES_BENCHMARK_COHORTS_CODE>


,benchmark_level_used,rows,confident_rows,low_confidence_rows,duplicate_keys,min_p50,max_p50
0,phase_indication_rare,659,105,554,0,1.0,277.0
1,phase_only,4,4,0,0,5.0,28.0
2,phase_ta,76,48,28,0,1.0,75.0
3,phase_ta_rare,138,53,85,0,1.0,78.0


Total in-memory artifact rows: 877
Duplicate benchmark keys: 0


#### <REF:SITES_BENCHMARK_PERCENTILES>
> #### **9. Benchmark Percentiles**
>
> Inspect P25/P50/P75/P90 rows. The production benchmark uses percentiles without winsorization because the upper tail is meaningful and large.


In [20]:
# <REF:SITES_BENCHMARK_PERCENTILES_CODE>
percentile_columns = [
    "benchmark_level_used",
    "benchmark_key",
    "benchmark_n",
    "benchmark_p25",
    "benchmark_p50",
    "benchmark_p75",
    "benchmark_p90",
    "low_confidence_flag",
]

print("Confident phase-only rows:")
display(
    in_memory_artifact.loc[in_memory_artifact["benchmark_level_used"].eq("phase_only"), percentile_columns]
    .sort_values("benchmark_key")
    .reset_index(drop=True)
)

print("Largest P90 cohorts:")
display(
    in_memory_artifact.sort_values("benchmark_p90", ascending=False)
    [percentile_columns]
    .head(20)
    .reset_index(drop=True)
)
# <REF:/SITES_BENCHMARK_PERCENTILES_CODE>


Confident phase-only rows:


,benchmark_level_used,benchmark_key,benchmark_n,benchmark_p25,benchmark_p50,benchmark_p75,benchmark_p90,low_confidence_flag
0,phase_only,phase_only|phase=PHASE1/PHASE2,1891,1.0,5.0,14.00,27.0,False
1,phase_only,phase_only|phase=PHASE2,8814,2.0,11.0,29.00,57.0,False
2,phase_only,phase_only|phase=PHASE2/PHASE3,500,3.0,14.0,43.25,81.1,False
3,phase_only,phase_only|phase=PHASE3,8675,6.0,28.0,72.00,140.0,False


Largest P90 cohorts:


,benchmark_level_used,benchmark_key,benchmark_n,benchmark_p25,benchmark_p50,benchmark_p75,benchmark_p90,low_confidence_flag
0,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,104,14.50,52.5,243.00,828.1,False
1,phase_indication_rare,phase_indication_rare|phase=PHASE2/PHASE3|indi...,4,43.25,242.0,446.50,476.2,True
2,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,14,14.75,42.5,82.00,430.6,True
3,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,100,26.50,85.0,226.75,372.7,False
4,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,13,44.00,164.0,228.00,324.4,True
5,phase_ta_rare,phase_ta_rare|phase=PHASE3|ta=CARDIOVASCULAR|r...,492,2.00,27.0,94.00,318.8,False
6,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,13,25.00,98.0,121.00,297.8,True
7,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,3,153.50,277.0,288.00,294.6,True
8,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,28,74.75,103.5,198.25,271.9,True
9,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,35,5.00,18.0,76.00,271.0,True


#### <REF:SITES_BENCHMARK_ARTIFACT>
> #### **10. Compact Benchmark Artifact**
>
> Load the S2 artifact from `frontend/data/site_benchmarks_v1.csv` and the practical report JSON. Production runtime should use this compact artifact, not the full historical source dataframe.


In [21]:
# <REF:SITES_BENCHMARK_ARTIFACT_CODE>
if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Missing benchmark artifact: {ARTIFACT_PATH}. Run python scripts/build_site_benchmarks.py")
if not REPORT_PATH.exists():
    raise FileNotFoundError(f"Missing benchmark report: {REPORT_PATH}. Run python scripts/build_site_benchmarks.py")

artifact = pd.read_csv(ARTIFACT_PATH)
report = json.loads(REPORT_PATH.read_text())

print(f"Artifact rows: {len(artifact):,}")
print("Artifact columns:")
print(list(artifact.columns))
print("Report summary:")
for key in [
    "source_records_loaded",
    "completed_positive_site_count_targets",
    "artifact_rows",
    "minimum_confident_cohort_threshold",
    "low_confidence_benchmark_rows",
    "duplicate_benchmark_keys",
    "phase_only_fallback_rows",
    "coverage_qa_not_available",
    "coverage_qa_low_confidence_matches",
    "source_data_version",
]:
    print(f"{key}: {report.get(key)}")

print("Rows by level from report:")
print(json.dumps(report["benchmark_rows_by_level"], indent=2))
# <REF:/SITES_BENCHMARK_ARTIFACT_CODE>


Artifact rows: 877
Artifact columns:
['benchmark_version', 'source_data_version', 'benchmark_key', 'phase', 'gbd_cause_id_3_ml', 'therapeutic_area', 'rare_disease_flag', 'benchmark_level_used', 'benchmark_n', 'benchmark_p25', 'benchmark_p50', 'benchmark_p75', 'benchmark_p90', 'low_confidence_flag', 'created_at', 'outlier_policy', 'calibration_notes']
Report summary:
source_records_loaded: 34066
completed_positive_site_count_targets: 19880
artifact_rows: 877
minimum_confident_cohort_threshold: 50
low_confidence_benchmark_rows: 667
duplicate_benchmark_keys: 0
phase_only_fallback_rows: 4
coverage_qa_not_available: 0
coverage_qa_low_confidence_matches: 0
source_data_version: 0a97519bd78f561a
Rows by level from report:
{
  "phase_indication_rare": 659,
  "phase_only": 4,
  "phase_ta": 76,
  "phase_ta_rare": 138
}


#### <REF:SITES_CLASSIFICATION>
> #### **11. Site-Count Status Classification**
>
> Demonstrate deterministic classification around P25, P75, and P90 boundaries. Boundaries mirror enrollment: P25 and P75 are inclusive `typical`; P90 is inclusive `ambitious`.


In [22]:
# <REF:SITES_CLASSIFICATION_CODE>
from src.site_benchmarks import classify_site_count

boundary_row = pd.Series({"benchmark_p25": 25, "benchmark_p50": 50, "benchmark_p75": 75, "benchmark_p90": 90})
classification_demo = pd.DataFrame({
    "planned_sites": [None, 0, 24, 25, 50, 75, 76, 90, 91],
})
classification_demo["site_count_status"] = classification_demo["planned_sites"].apply(lambda value: classify_site_count(value, boundary_row))
display(classification_demo)
# <REF:/SITES_CLASSIFICATION_CODE>


,planned_sites,site_count_status
0,NaN,not_available
1,0.0,not_available
2,24.0,below_benchmark
3,25.0,typical
4,50.0,typical
5,75.0,typical
6,76.0,ambitious
7,90.0,ambitious
8,91.0,above_benchmark_high


#### <REF:SITES_RUNTIME_LOOKUP_DEMO>
> #### **12. Runtime Lookup Demo**
>
> Demonstrate how production runtime can use only the current trial snapshot and compact artifact. The fallback order is phase + indication + rare, then phase + therapeutic area + rare, then phase + therapeutic area, then phase only. Confident rows are preferred; low-confidence rows are a last resort.


In [23]:
# <REF:SITES_RUNTIME_LOOKUP_DEMO_CODE>
from src.site_benchmarks import (
    load_site_benchmarks,
    lookup_site_benchmark,
    planned_sites_metadata,
)

runtime_artifact = load_site_benchmarks(ARTIFACT_PATH)
print(f"Runtime artifact rows: {len(runtime_artifact):,}")

strict_row = runtime_artifact[
    runtime_artifact["benchmark_level_used"].eq("phase_indication_rare")
    & runtime_artifact["low_confidence_flag"].eq(False)
].iloc[0]
strict_snapshot = {
    "phase": strict_row["phase"],
    "gbd_cause_id_3_ml": int(strict_row["gbd_cause_id_3_ml"]),
    "therapeutic_area": strict_row.get("therapeutic_area"),
    "is_rare_disease_ml": int(strict_row["rare_disease_flag"]),
}
strict_lookup = lookup_site_benchmark(strict_snapshot, runtime_artifact)
print("Strict snapshot:")
print(json.dumps(strict_snapshot, indent=2))
print("Strict lookup level:", strict_lookup["benchmark_level_used"])
display(pd.DataFrame([strict_lookup]))

fallback_row = runtime_artifact[
    runtime_artifact["benchmark_level_used"].eq("phase_ta")
    & runtime_artifact["low_confidence_flag"].eq(False)
].iloc[0]
fallback_snapshot = {
    "phase": fallback_row["phase"],
    "gbd_cause_id_3_ml": 999999999,
    "therapeutic_area": fallback_row["therapeutic_area"],
    "is_rare_disease_ml": 1,
}
fallback_lookup = lookup_site_benchmark(fallback_snapshot, runtime_artifact)
print("Fallback snapshot:")
print(json.dumps(fallback_snapshot, indent=2))
print("Fallback lookup level:", fallback_lookup["benchmark_level_used"] if fallback_lookup is not None else None)
display(pd.DataFrame([fallback_lookup]))
# <REF:/SITES_RUNTIME_LOOKUP_DEMO_CODE>


Runtime artifact rows: 877
Strict snapshot:
{
  "phase": "PHASE1/PHASE2",
  "gbd_cause_id_3_ml": 426,
  "therapeutic_area": null,
  "is_rare_disease_ml": 0
}
Strict lookup level: phase_indication_rare


,benchmark_version,source_data_version,benchmark_key,phase,gbd_cause_id_3_ml,therapeutic_area,rare_disease_flag,benchmark_level_used,benchmark_n,benchmark_p25,benchmark_p50,benchmark_p75,benchmark_p90,low_confidence_flag,created_at,outlier_policy,calibration_notes
26,site_benchmarks_v1,0a97519bd78f561a,phase_indication_rare|phase=PHASE1/PHASE2|indi...,PHASE1/PHASE2,426.0,None,0.0,phase_indication_rare,64,4.75,14.5,24.5,46.4,False,2026-06-02T14:15:02+00:00,positive completed registry-derived facility-c...,Deterministic S2 planned-sites benchmark based...


Fallback snapshot:
{
  "phase": "PHASE1/PHASE2",
  "gbd_cause_id_3_ml": 999999999,
  "therapeutic_area": "DERMATOLOGY",
  "is_rare_disease_ml": 1
}
Fallback lookup level: phase_ta


,benchmark_version,source_data_version,benchmark_key,phase,gbd_cause_id_3_ml,therapeutic_area,rare_disease_flag,benchmark_level_used,benchmark_n,benchmark_p25,benchmark_p50,benchmark_p75,benchmark_p90,low_confidence_flag,created_at,outlier_policy,calibration_notes
665,site_benchmarks_v1,0a97519bd78f561a,phase_ta|phase=PHASE1/PHASE2|ta=DERMATOLOGY,PHASE1/PHASE2,NaN,DERMATOLOGY,NaN,phase_ta,102,1.0,2.0,6.0,14.0,False,2026-06-02T14:15:02+00:00,positive completed registry-derived facility-c...,Deterministic S2 planned-sites benchmark based...


#### <REF:SITES_SNAPSHOT_METADATA>
> #### **13. Planned Sites Metadata Object**
>
> This is the S2 runtime utility output shape for future snapshot use. S2 does not attach it to Simulation Mode and does not activate `planned_sites` in the UI.


In [24]:
# <REF:SITES_SNAPSHOT_METADATA_CODE>
metadata = planned_sites_metadata(
    strict_snapshot,
    strict_lookup["benchmark_p50"],
    source="registry_facility_count_proxy",
    artifact=runtime_artifact,
)
print(json.dumps(metadata, indent=2))
# <REF:/SITES_SNAPSHOT_METADATA_CODE>


{
  "planned_sites": {
    "value": 14.5,
    "source": "registry_facility_count_proxy",
    "benchmark_level_used": "phase_indication_rare",
    "benchmark_n": 64,
    "benchmark_p25": 4.75,
    "benchmark_p50": 14.5,
    "benchmark_p75": 24.5,
    "benchmark_p90": 46.4,
    "site_count_status": "typical",
    "support_level": "not_evaluated",
    "supporting_signals": [],
    "conflicting_signals": [],
    "benchmark_snapshot_id": "site_benchmarks_v1:0a97519bd78f561a:phase_indication_rare|phase=PHASE1/PHASE2|indication=426|rare=0",
    "is_benchmark_stale": false,
    "low_confidence_flag": false,
    "interpretation_hint": "Site count is within the usual completed registry facility-count proxy benchmark range for the matched cohort."
  }
}


#### <REF:SITES_ENROLLMENT_ALIGNMENT>
> #### **14. Enrollment/Site Realism: Patients Per Site Sanity Check**
>
> This answers the practical question: how do we make sure a future site-count assumption for an ongoing trial is realistic relative to the planned enrollment number?
>
> S2 intentionally does **not** couple `planned_sites` to `planned_enrollment` in runtime. The primary site benchmark stands on its own facility-count proxy distribution first. The anticipated future coherence check is a secondary signal: calculate `planned_enrollment / planned_sites` and compare that patients-per-site value with historical completed-trial patients-per-site proxy ranges for the same fallback cohort.
>
> Caveat: this is notebook-only exploration. It does not activate a support/conflict signal, Coherence Score, LLM narrative, UI warning, or `/predict` behavior.


In [25]:
# <REF:SITES_ENROLLMENT_ALIGNMENT_CODE>
pps_source_base = benchmark_source.copy()
if "enrollment" not in pps_source_base.columns:
    pps_source_base = pps_source_base.merge(
        df[["nct_id", "enrollment"]],
        on="nct_id",
        how="left",
        validate="one_to_one",
    )
pps_source_base["enrollment"] = pd.to_numeric(pps_source_base["enrollment"], errors="coerce")
pps_source = pps_source_base[
    pps_source_base["is_completed_positive_site_count_target"]
    & pps_source_base["enrollment"].gt(0)
].copy()
pps_source["patients_per_site_proxy"] = pps_source["enrollment"] / pps_source["number_of_facilities"]

pps_quality = {
    "completed_positive_enrollment_and_site_rows": int(len(pps_source)),
    "p25": round(float(pps_source["patients_per_site_proxy"].quantile(0.25)), 2),
    "median": round(float(pps_source["patients_per_site_proxy"].quantile(0.50)), 2),
    "p75": round(float(pps_source["patients_per_site_proxy"].quantile(0.75)), 2),
    "p90": round(float(pps_source["patients_per_site_proxy"].quantile(0.90)), 2),
    "p95": round(float(pps_source["patients_per_site_proxy"].quantile(0.95)), 2),
    "max": round(float(pps_source["patients_per_site_proxy"].max()), 2),
}
print("Global completed-trial patients-per-site proxy summary:")
print(json.dumps(pps_quality, indent=2))

def pps_percentiles_for_snapshot(snapshot: dict, source: pd.DataFrame, min_n: int = 50) -> tuple[str, dict[str, float | int] | None]:
    phase = str(snapshot.get("phase", "")).strip().upper()
    indication = pd.to_numeric(snapshot.get("gbd_cause_id_3_ml"), errors="coerce")
    ta = str(snapshot.get("therapeutic_area", "")).strip().upper()
    rare = pd.to_numeric(snapshot.get("is_rare_disease_ml"), errors="coerce")

    candidates = []
    if phase and pd.notna(indication) and pd.notna(rare):
        candidates.append((
            "phase_indication_rare",
            source[
                source["phase"].eq(phase)
                & source["gbd_cause_id_3_ml"].eq(int(indication))
                & source["is_rare_disease_ml"].eq(int(rare))
            ],
        ))
    if phase and ta and pd.notna(rare):
        candidates.append((
            "phase_ta_rare",
            source[
                source["phase"].eq(phase)
                & source["therapeutic_area"].eq(ta)
                & source["is_rare_disease_ml"].eq(int(rare))
            ],
        ))
    if phase and ta:
        candidates.append(("phase_ta", source[source["phase"].eq(phase) & source["therapeutic_area"].eq(ta)]))
    if phase:
        candidates.append(("phase_only", source[source["phase"].eq(phase)]))

    first_available = None
    for level, cohort in candidates:
        if cohort.empty:
            continue
        q = cohort["patients_per_site_proxy"].quantile([0.25, 0.50, 0.75, 0.90])
        row = {
            "benchmark_n": int(len(cohort)),
            "pps_p25": round(float(q.loc[0.25]), 2),
            "pps_p50": round(float(q.loc[0.50]), 2),
            "pps_p75": round(float(q.loc[0.75]), 2),
            "pps_p90": round(float(q.loc[0.90]), 2),
            "low_confidence_flag": bool(len(cohort) < min_n),
        }
        if first_available is None:
            first_available = (level, row)
        if len(cohort) >= min_n:
            return level, row
    return first_available if first_available is not None else ("not_available", None)

def classify_patients_per_site(planned_enrollment, planned_sites, pps_row):
    enrollment_value = pd.to_numeric(planned_enrollment, errors="coerce")
    site_value = pd.to_numeric(planned_sites, errors="coerce")
    if pd.isna(enrollment_value) or pd.isna(site_value) or float(enrollment_value) <= 0 or float(site_value) <= 0 or pps_row is None:
        return {"patients_per_site": None, "pps_status": "not_available"}
    pps = float(enrollment_value) / float(site_value)
    if pps < pps_row["pps_p25"]:
        status = "low_patients_per_site"
    elif pps <= pps_row["pps_p75"]:
        status = "typical_patients_per_site"
    elif pps <= pps_row["pps_p90"]:
        status = "high_patients_per_site"
    else:
        status = "very_high_patients_per_site"
    return {"patients_per_site": round(pps, 2), "pps_status": status}

pps_level, pps_row = pps_percentiles_for_snapshot(strict_snapshot, pps_source, min_n=MIN_N)
print("Matched patients-per-site proxy benchmark level:", pps_level)
print(json.dumps(pps_row, indent=2))

alignment_examples = pd.DataFrame([
    {"planned_enrollment": 300, "planned_sites": 60},
    {"planned_enrollment": 600, "planned_sites": 60},
    {"planned_enrollment": 1200, "planned_sites": 60},
    {"planned_enrollment": 1200, "planned_sites": 15},
])
alignment_results = alignment_examples.apply(
    lambda row: classify_patients_per_site(row["planned_enrollment"], row["planned_sites"], pps_row),
    axis=1,
    result_type="expand",
)
display(pd.concat([alignment_examples, alignment_results], axis=1))

print("Interpretation: future S3+ can use this as a secondary coherence/support signal, but S2 intentionally does not activate it.")
# <REF:/SITES_ENROLLMENT_ALIGNMENT_CODE>


Global completed-trial patients-per-site proxy summary:
{
  "completed_positive_enrollment_and_site_rows": 19868,
  "p25": 4.13,
  "median": 8.57,
  "p75": 28.62,
  "p90": 126.0,
  "p95": 272.82,
  "max": 90116.0
}
Matched patients-per-site proxy benchmark level: phase_indication_rare
{
  "benchmark_n": 64,
  "pps_p25": 3.21,
  "pps_p50": 5.69,
  "pps_p75": 13.55,
  "pps_p90": 33.4,
  "low_confidence_flag": false
}


,planned_enrollment,planned_sites,patients_per_site,pps_status
0,300,60,5.0,typical_patients_per_site
1,600,60,10.0,typical_patients_per_site
2,1200,60,20.0,high_patients_per_site
3,1200,15,80.0,very_high_patients_per_site


Interpretation: future S3+ can use this as a secondary coherence/support signal, but S2 intentionally does not activate it.


#### <REF:SITES_VALIDATION>
> #### **15. Validation and Audit Summary**
>
> Summarize practical checks: required columns, artifact/report consistency, fallback behavior, boundary classification, and generated report values.
>
> S2 validation commands:
>
> ```bash
> python scripts/build_site_benchmarks.py
> python scripts/check_site_benchmarks.py
> python -m py_compile scripts/build_site_benchmarks.py scripts/check_site_benchmarks.py src/site_benchmarks.py
> git diff --check
> ```


In [26]:
# <REF:SITES_VALIDATION_CODE>
required_artifact_columns = {
    "benchmark_version", "source_data_version", "benchmark_key", "phase", "gbd_cause_id_3_ml",
    "therapeutic_area", "rare_disease_flag", "benchmark_level_used", "benchmark_n",
    "benchmark_p25", "benchmark_p50", "benchmark_p75", "benchmark_p90", "low_confidence_flag", "created_at",
    "outlier_policy", "calibration_notes",
}
missing_artifact_columns = sorted(required_artifact_columns.difference(artifact.columns))
print("Missing artifact columns:", missing_artifact_columns)
assert not missing_artifact_columns

print("Report JSON consistency:")
print(f"Artifact rows: {len(artifact):,}")
print(f"Report artifact rows: {report['artifact_rows']:,}")
print(f"Report low-confidence rows: {report['low_confidence_benchmark_rows']:,}")
print(f"Artifact low-confidence rows: {int(artifact['low_confidence_flag'].astype(str).str.lower().isin(['true', '1', 'yes']).sum()):,}")
print(f"Duplicate benchmark keys: {int(artifact['benchmark_key'].duplicated().sum()):,}")
assert len(artifact) == report["artifact_rows"]
assert int(artifact["benchmark_key"].duplicated().sum()) == report["duplicate_benchmark_keys"]

print("Coverage QA match counts:")
print(json.dumps(report["coverage_qa_match_counts"], indent=2))
assert report["coverage_qa_not_available"] == 0
assert report["coverage_qa_low_confidence_matches"] == 0

print("Sparse and low-confidence summary:")
display(
    artifact.assign(
        low_confidence_flag_clean=artifact["low_confidence_flag"].astype(str).str.lower().isin(["true", "1", "yes"])
    )
    .groupby("benchmark_level_used")
    .agg(
        rows=("benchmark_key", "size"),
        low_confidence_rows=("low_confidence_flag_clean", "sum"),
        median_n=("benchmark_n", "median"),
    )
    .reset_index()
)

print("Runtime missing-artifact behavior:")
with tempfile.TemporaryDirectory() as tmpdir:
    missing_metadata = planned_sites_metadata(strict_snapshot, 25, artifact_path=Path(tmpdir) / "missing.csv")
print(json.dumps(missing_metadata, indent=2))
assert missing_metadata["planned_sites"]["site_count_status"] == "not_available"

print("S2 validation summary: artifact schema, report consistency, fallback coverage, runtime lookup, and missing-artifact behavior passed in notebook.")
# <REF:/SITES_VALIDATION_CODE>


Missing artifact columns: []
Report JSON consistency:
Artifact rows: 877
Report artifact rows: 877
Report low-confidence rows: 667
Artifact low-confidence rows: 667
Duplicate benchmark keys: 0
Coverage QA match counts:
{
  "phase_indication_rare": 23439,
  "phase_only": 1136,
  "phase_ta": 2231,
  "phase_ta_rare": 7260
}
Sparse and low-confidence summary:


,benchmark_level_used,rows,low_confidence_rows,median_n
0,phase_indication_rare,659,554,7.0
1,phase_only,4,0,5283.0
2,phase_ta,76,28,95.0
3,phase_ta_rare,138,85,37.0


Runtime missing-artifact behavior:
{
  "planned_sites": {
    "value": 25.0,
    "source": "registry_facility_count_proxy",
    "benchmark_level_used": "not_available",
    "benchmark_n": null,
    "benchmark_p25": null,
    "benchmark_p50": null,
    "benchmark_p75": null,
    "benchmark_p90": null,
    "site_count_status": "not_available",
    "support_level": "not_evaluated",
    "supporting_signals": [],
    "conflicting_signals": [],
    "benchmark_snapshot_id": null,
    "is_benchmark_stale": false,
    "low_confidence_flag": true,
    "interpretation_hint": "Site-count benchmark is not available for this snapshot."
  }
}
S2 validation summary: artifact schema, report consistency, fallback coverage, runtime lookup, and missing-artifact behavior passed in notebook.


#### <REF:SITES_CONCLUSION_AND_NEXT_STEPS>
> #### **16. Conclusion and Next Steps**
>
> S1B and S2 are now reproducible here:
>
> - S1B source contract and feasibility audit.
> - Offline benchmark artifact builder.
> - Compact production-friendly site-count artifact.
> - Runtime lookup utility.
> - Validation/reporting checks.
> - Notebook-only patients-per-site realism check for future coherence work.
>
> Later S3 work, only after separate authorization:
>
> - Add `planned_sites` to Simulation Mode.
> - Attach benchmark metadata to prediction snapshots.
> - Decide how to expose patients-per-site support/conflict signals.
> - Keep site count outside XGBoost, SHAP, `/predict`, calibration, and API contracts unless a separate architecture decision changes that boundary.
>
> Still excluded: `planned_duration_months`, `planned_countries`, cost, market, recruitment modelling, Coherence Score implementation, LLM calls, and deployment changes.


In [27]:
# <REF:SITES_CONCLUSION_AND_NEXT_STEPS_CODE>
print("Reproducible artifact:", ARTIFACT_PATH.relative_to(project_root))
print("Runtime utility:", RUNTIME_UTILITY_PATH.relative_to(project_root))
print("Primary rebuild command: python scripts/build_site_benchmarks.py")
print("Primary validation command: python scripts/check_site_benchmarks.py")
print("S2 notebook complete; planned_sites remains inactive until separately authorized S3 UI/snapshot integration.")
# <REF:/SITES_CONCLUSION_AND_NEXT_STEPS_CODE>


Reproducible artifact: frontend/data/site_benchmarks_v1.csv
Runtime utility: src/site_benchmarks.py
Primary rebuild command: python scripts/build_site_benchmarks.py
Primary validation command: python scripts/check_site_benchmarks.py
S2 notebook complete; planned_sites remains inactive until separately authorized S3 UI/snapshot integration.
